In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

X_train = pd.read_csv('train.csv')
X_test = pd.read_csv('test.csv')

In [19]:
# Удаляем признаки, которые не являются релевантными 

to_drop = ['PassengerId', 'Name',]
 
X_train.drop(to_drop, axis=1, inplace=True)
X_test.drop(to_drop, axis=1, inplace=True)

y = X_train['Survived'] # сохраняем целевую переменную, как отдельный вектор

X_train.drop('Survived', axis=1, inplace=True) # также удаляем целевую переменную из таблицы

In [20]:
# Делю признаки на числовые и категориальные
num_feats = X_train.select_dtypes(include=[int, float, ]).columns.difference(['Pclass']).tolist()
cat_feats = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
cat_feats.append('Pclass')


print(f"Числовые колонки: {num_feats}")
print(f"Категориальные колонки: {cat_feats}")

Числовые колонки: ['Age', 'Fare', 'Parch', 'SibSp']
Категориальные колонки: ['Sex', 'Ticket', 'Cabin', 'Embarked', 'Pclass']


In [21]:
# Заполняю пропуски средним по столбцу Age
mean_train = X_train['Age'].mean()
X_train['Age'] = X_train['Age'].fillna(mean_train)

mean_test = X_test['Age'].mean()
X_test['Age'] = X_test['Age'].fillna(mean_test)

print(f'Признак Age:\n')
print(f'в train заполнен значением {mean_train}')
print(f'в train заполнен значением {mean_test}')

Признак Age:

в train заполнен значением 29.69911764705882
в train заполнен значением 30.272590361445783


In [22]:
# Заполненяю пропуски модой по столбцу Cabin
mode_train = X_train['Cabin'].mode()[0] if not X_train['Cabin'].mode().empty else 'Unknown'
X_train['Cabin'] = X_train['Cabin'].fillna(mode_train)

mode_test = X_test['Cabin'].mode()[0] if not X_test['Cabin'].mode().empty else 'Unknown'
X_test['Cabin'] = X_test['Cabin'].fillna(mode_test)

print(f'Признак Cabin:\n')
print(f"в train признак заполнен модой {mode_train}")
print(f"в test признак заполнен модой {mode_test}")



Признак Cabin:

в train признак заполнен модой B96 B98
в test признак заполнен модой B57 B59 B63 B66


In [23]:
fare_median = X_train['Fare'].median() 
X_test['Fare'] = X_test['Fare'].fillna(fare_median)

In [24]:
# После обработки пропусков

print(X_train.isnull().sum(), '\n\n')

print(X_test.isnull().sum())

Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Ticket      0
Fare        0
Cabin       0
Embarked    2
dtype: int64 


Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Ticket      0
Fare        0
Cabin       0
Embarked    0
dtype: int64


In [25]:
# Определяем функцию межквартильного разамаха для всех числовых признаков
def igr(num_feat:str):
    Q1 = X_train[num_feat].quantile(0.25) # расчет 25-го процентиля в столбце, при этом сначала столбец монотонно упорядочивается на время расчета
    Q3 = X_train[num_feat].quantile(0.75)

    IQR = Q3 - Q1 # размах

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers_lower = X_train[X_train[num_feat] < lower_bound]
    count_lower = len(outliers_lower)
    
    outliers_upper = X_train[X_train[num_feat] > upper_bound]
    count_upper = len(outliers_upper)
    
    total_outliers = count_lower + count_upper
    
    return round(total_outliers / len(X_train) * 100, 2), lower_bound, upper_bound

In [26]:
# Теперь устраним выбросы

for column in num_feats:
    outliers, lower, upper = igr(column) # outliers -- процент выбросов для признака
    print(f"{column}: {outliers}% выбросов, границы: [{lower:.2f}, {upper:.2f}]")
    if outliers < 1:
        continue

    elif 1 <= outliers < 5:
        X_train[column] = X_train[column].clip(lower, upper) # замена на min или max допустимое значение
        X_test[column] = X_test[column].clip(lower, upper) 

    elif 5 <= outliers < 15:
        median_val = X_train[column].median()
        outlier_mask = (X_train[column] < lower) | (X_train[column] > upper)
        X_train.loc[outlier_mask, column] = median_val

        outlier_mask_test = (X_test[column] < lower) | (X_test[column] > upper)
        X_test.loc[outlier_mask_test, column] = median_val

    elif column == 'Parch':

            # Создаем бинарный признак "есть родственники"
            X_train['Parch_has'] = (X_train[column] > 0).astype(int)
            X_test['Parch_has'] = (X_test[column] > 0).astype(int)
            
            # Создаем категории: 0, 1, 2, 3+
            X_train['Parch_cat'] = X_train[column].clip(upper=3)
            X_test['Parch_cat'] = X_test[column].clip(upper=3)

            X_train.drop(['Parch'], axis=1, inplace=True)
            X_test.drop(['Parch'], axis=1, inplace=True)
            continue

    elif 15 <= outliers < 30:
        # Логарифмическое преобразование
        min_val = X_train[column].min()
        shift = abs(min_val) + 1 if min_val <= 0 else 0
        
        X_train[column] = np.log1p(X_train[column] + shift)
        X_test[column] = np.log1p(X_test[column] + shift)


    outliers_lower = X_train[X_train[column] < lower]
    count_lower = len(outliers_lower)
    
    outliers_upper = X_train[X_train[column] > upper]
    count_upper = len(outliers_upper)
    
    new_outliers = count_lower + count_upper   
    print(f"после обработки {column}: {new_outliers}% выбросов, границы: [{lower:.2f}, {upper:.2f}]")

Age: 7.41% выбросов, границы: [2.50, 54.50]
после обработки Age: 0% выбросов, границы: [2.50, 54.50]
Fare: 13.02% выбросов, границы: [-26.72, 65.63]
после обработки Fare: 0% выбросов, границы: [-26.72, 65.63]
Parch: 23.91% выбросов, границы: [0.00, 0.00]
SibSp: 5.16% выбросов, границы: [-1.50, 2.50]
после обработки SibSp: 0% выбросов, границы: [-1.50, 2.50]


In [27]:
def preprocess_categorical(train, test):
    
    # Кодируем признак "пол" через LabelEncoder
    le = LabelEncoder()
    X_train['Sex'] = le.fit_transform(X_train['Sex'])
    X_test['Sex'] = le.transform(X_test['Sex'])

    # Заполняем пропуски EMBARKED модой из тренировочных данных
    embarked_mode = train['Embarked'].mode()[0]
    train['Embarked'] = train['Embarked'].fillna(embarked_mode)
    
    # CABIN - извлекаем информацию
    # Буква каюты (дек) и наличие каюты
    for df in [train, test]:
        df['Cabin_letter'] = df['Cabin'].str[0]  # Первая буква
        df['Cabin_letter'] = df['Cabin_letter'].fillna('Unknown')
        # Объединяем редкие категории кают
        cabin_counts = df['Cabin_letter'].value_counts()
        rare_cabins = cabin_counts[cabin_counts < 10].index
        df['Cabin_letter'] = df['Cabin_letter'].replace(rare_cabins, 'Other')
    
    # TICKET - извлекаем информацию
    for df in [train, test]:
        # Тип билета (буквы в начале)
        df['Ticket_prefix'] = df['Ticket'].str.extract(r'([A-Za-z\.\/]+)')[0]
        df['Ticket_prefix'] = df['Ticket_prefix'].fillna('None')
        # Объединяем редкие префиксы
        prefix_counts = df['Ticket_prefix'].value_counts()
        rare_prefixes = prefix_counts[prefix_counts < 5].index
        df['Ticket_prefix'] = df['Ticket_prefix'].replace(rare_prefixes, 'Rare')
    
    # Удаляем исходные колонки
    cols_to_drop = ['Cabin', 'Ticket']
    for df in [train, test]:
        df.drop(cols_to_drop, axis=1, inplace=True, errors='ignore')
    
    return train, test
X_train, X_test = preprocess_categorical(X_train, X_test)

In [28]:
new_cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Ищем продходящий метод кодировки категориальных признаков
for df in [X_train, X_test]:
        for col in new_cat_cols:
            nunique = df[col].nunique()
            if nunique <= 15:
                 print(f"Для {col} уникальных {nunique} -> используем one-hot encoding")
            elif 15 < nunique:
                print(f"Для {col} уникальных {nunique} -> используем target encoding")
                  

Для Embarked уникальных 3 -> используем one-hot encoding
Для Cabin_letter уникальных 7 -> используем one-hot encoding
Для Ticket_prefix уникальных 15 -> используем one-hot encoding
Для Embarked уникальных 3 -> используем one-hot encoding
Для Cabin_letter уникальных 4 -> используем one-hot encoding
Для Ticket_prefix уникальных 8 -> используем one-hot encoding


In [29]:
# Остальные через one-hot encoder
def OHE_categorical(train, test):
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
    ohe.fit(train[new_cat_cols])
    encoded_cols = ohe.get_feature_names_out(new_cat_cols)

    for df in [train, test]:
        # Трансформируем и сразу создаем DataFrame
        encoded_df = pd.DataFrame(ohe.transform(df[new_cat_cols]),
                                  columns=encoded_cols,
                                  index=df.index) 
        
        # Объединяем: удаляем старые и добавляем новые колонки
        df.drop(new_cat_cols, axis=1, inplace=True)
        df[encoded_cols] = encoded_df
    return encoded_cols

            
            
ohe_columns = OHE_categorical(X_train, X_test)

In [30]:
X_test.dtypes

Pclass                        int64
Sex                           int64
Age                         float64
SibSp                         int64
Fare                        float64
Parch_has                     int64
Parch_cat                     int64
Embarked_Q                  float64
Embarked_S                  float64
Cabin_letter_B              float64
Cabin_letter_C              float64
Cabin_letter_D              float64
Cabin_letter_E              float64
Cabin_letter_F              float64
Cabin_letter_Other          float64
Ticket_prefix_C             float64
Ticket_prefix_C.A.          float64
Ticket_prefix_CA            float64
Ticket_prefix_CA.           float64
Ticket_prefix_F.C.C.        float64
Ticket_prefix_None          float64
Ticket_prefix_PC            float64
Ticket_prefix_Rare          float64
Ticket_prefix_S.O.C.        float64
Ticket_prefix_SC/PARIS      float64
Ticket_prefix_SOTON/O.Q.    float64
Ticket_prefix_SOTON/OQ      float64
Ticket_prefix_STON/O        

In [31]:
new_num_cols = X_train.select_dtypes(include=[float, int]).columns.tolist()

# Вычисляем дисперсии по каждому признаку
numeric = X_train[new_num_cols].values
variances = np.var(numeric, axis=0, ddof=0) 
print("Дисперсии признаков:")
for col, var in zip(new_num_cols, variances):
    print(f"  {col}: {var:.2f}")

Дисперсии признаков:
  Pclass: 0.70
  Sex: 0.23
  Age: 95.94
  SibSp: 0.27
  Fare: 161.44
  Parch_has: 0.18
  Parch_cat: 0.51
  Embarked_Q: 0.08
  Embarked_S: 0.20
  Cabin_letter_B: 0.15
  Cabin_letter_C: 0.06
  Cabin_letter_D: 0.04
  Cabin_letter_E: 0.03
  Cabin_letter_F: 0.01
  Cabin_letter_Other: 0.01
  Ticket_prefix_C: 0.01
  Ticket_prefix_C.A.: 0.03
  Ticket_prefix_CA: 0.01
  Ticket_prefix_CA.: 0.01
  Ticket_prefix_F.C.C.: 0.01
  Ticket_prefix_None: 0.19
  Ticket_prefix_PC: 0.06
  Ticket_prefix_Rare: 0.05
  Ticket_prefix_S.O.C.: 0.01
  Ticket_prefix_SC/PARIS: 0.01
  Ticket_prefix_SOTON/O.Q.: 0.01
  Ticket_prefix_SOTON/OQ: 0.01
  Ticket_prefix_STON/O: 0.02
  Ticket_prefix_W./C.: 0.01


In [32]:
# Проверяем необходимо ли масштабирование
max_var = variances.max()
min_var = variances.min()
ratio = max_var / min_var

print(f"\nОтношение дисперсий: {ratio:.1f}")

if ratio >= 10:
    print("=> Необходимо маштабирование")


Отношение дисперсий: 28930.9
=> Необходимо маштабирование


In [33]:
# Используем standart scaler для масштабирования

scaler = StandardScaler()

scaler.fit(X_train,)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train = pd.DataFrame(X_train_scaled,
                       columns=X_train.columns, 
                       index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, 
                      columns=X_test.columns, 
                      index=X_test.index)
X_train.describe()


,Pclass,Sex,Age,SibSp,Fare,Parch_has,Parch_cat,Embarked_Q,Embarked_S,Cabin_letter_B,...,Ticket_prefix_F.C.C.,Ticket_prefix_None,Ticket_prefix_PC,Ticket_prefix_Rare,Ticket_prefix_S.O.C.,Ticket_prefix_SC/PARIS,Ticket_prefix_SOTON/O.Q.,Ticket_prefix_SOTON/OQ,Ticket_prefix_STON/O,Ticket_prefix_W./C.
count,8.910000e+02,8.910000e+02,8.910000e+02,8.910000e+02,8.910000e+02,8.910000e+02,8.910000e+02,891.000000,8.910000e+02,8.910000e+02,...,8.910000e+02,8.910000e+02,8.910000e+02,8.910000e+02,8.910000e+02,8.910000e+02,8.910000e+02,8.910000e+02,8.910000e+02,8.910000e+02
mean,-8.772133e-17,-1.156327e-16,3.269613e-16,8.572766e-17,-9.569599e-17,-1.036707e-16,-7.974666e-17,0.000000,-8.373399e-17,-3.389233e-17,...,1.395567e-17,4.784800e-17,-7.974666e-18,1.116453e-16,1.594933e-17,1.196200e-17,5.582266e-17,-8.971499e-18,-6.379733e-17,-5.681949e-17
std,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00,1.000562,1.000562e+00,1.000562e+00,...,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00
min,-1.566107e+00,-1.355574e+00,-2.648403e+00,-5.704719e-01,-1.368156e+00,-5.604991e-01,-5.067865e-01,-0.307562,-1.623803e+00,-2.162212e+00,...,-7.512217e-02,-1.695262e+00,-2.687046e-01,-2.279212e-01,-7.512217e-02,-7.512217e-02,-9.518415e-02,-8.898625e-02,-1.435916e-01,-1.010153e-01
25%,-3.693648e-01,-1.355574e+00,-5.299165e-01,-5.704719e-01,-7.455778e-01,-5.604991e-01,-5.067865e-01,-0.307562,-1.623803e+00,4.624894e-01,...,-7.512217e-02,-1.695262e+00,-2.687046e-01,-2.279212e-01,-7.512217e-02,-7.512217e-02,-9.518415e-02,-8.898625e-02,-1.435916e-01,-1.010153e-01
50%,8.273772e-01,7.376951e-01,7.746308e-02,-5.704719e-01,-2.305564e-01,-5.604991e-01,-5.067865e-01,-0.307562,6.158384e-01,4.624894e-01,...,-7.512217e-02,5.898793e-01,-2.687046e-01,-2.279212e-01,-7.512217e-02,-7.512217e-02,-9.518415e-02,-8.898625e-02,-1.435916e-01,-1.010153e-01
75%,8.273772e-01,7.376951e-01,4.144691e-01,1.347605e+00,5.325391e-01,-5.604991e-01,-5.067865e-01,-0.307562,6.158384e-01,4.624894e-01,...,-7.512217e-02,5.898793e-01,-2.687046e-01,-2.279212e-01,-7.512217e-02,-7.512217e-02,-9.518415e-02,-8.898625e-02,-1.435916e-01,-1.010153e-01
max,8.273772e-01,7.376951e-01,2.558480e+00,3.265682e+00,3.747586e+00,1.784124e+00,3.687147e+00,3.251373,6.158384e-01,4.624894e-01,...,1.331165e+01,5.898793e-01,3.721559e+00,4.387482e+00,1.331165e+01,1.331165e+01,1.050595e+01,1.123769e+01,6.964194e+00,9.899495e+00


In [ ]:
# Применение модели логистической регрессии 

lr = LogisticRegression()

lr.fit(X_train, y)
y_pred = lr.predict(X_test) # Score: 0.76076, Position: 11739

test_original = pd.read_csv('test.csv')
ids = test_original['PassengerId']

results = pd.DataFrame({
    'PassengerId': ids,
    'Survived': y_pred
})

results.to_csv('gender_submission.csv', index=False)